# 01 — Anatomy of a Foundry prompt agent

**Foundry feature:** what a **prompt agent** is made of — instructions, tools, an evaluation
dataset, and an evaluation contract — before you create one in the portal. **Mode: CODE** (reads
files already in the repo, no Foundry call).

In [ ]:
import json, subprocess, sys
from pathlib import Path

# Repo layout: this notebook lives in notebooks/, the pack lives in ../prompt-agent-optimizer-baselines
PACK_ROOT = Path("..").resolve() / "prompt-agent-optimizer-baselines"
AGENT_ID = "01-travel-approval-strict"          # <- the one case study every notebook in this series uses
AGENT_DIR = PACK_ROOT / AGENT_ID

assert AGENT_DIR.exists(), f"Can't find {AGENT_DIR} -- run this notebook from a checkout of the repo."

def run(cmd, cwd=PACK_ROOT):
    """Run a pack CLI tool and print its output, the way you would from a terminal."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

print(f"Pack root : {PACK_ROOT}")
print(f"Case study: {AGENT_ID}")

## `agent.yaml` — the agent definition

This is the top-level manifest: name, model, and pointers to the instructions and tool files. It's
also where the pack records metadata Foundry itself doesn't need (`baseline_quality`, `uses_mcp`,
`dataset_split`) that this notebook series uses to explain *why* this sample looks the way it does.

In [ ]:
print((AGENT_DIR / "agent.yaml").read_text())

## `instructions.md` — the system prompt Foundry's optimizer rewrites

This is the only file the Optimize wizard is allowed to change. Everything else (tool definitions,
the model) stays fixed across a run.

In [ ]:
print((AGENT_DIR / "instructions.md").read_text())

Read it with an eye for what notebook 05/06 will check automatically later: four numbered approval
tiers, hard-coded lodging caps, a restricted-destination hard stop, and exactly one hedged,
optional-sounding line near the end (the sign-off). That last line is this baseline's only real
flaw — see `../prompt-agent-optimizer-baselines/01-travel-approval-strict/README.md`.

## `tools.json` — the function tools Foundry exposes to the agent

Three function tools, each a standard JSON-schema tool definition — the same shape Foundry's agent
editor asks you to fill in by hand or paste as JSON.

In [ ]:
tools = json.loads((AGENT_DIR / "tools.json").read_text())
for t in tools:
    fn = t["function"]
    required = fn.get("parameters", {}).get("required", [])
    print(f"- {fn['name']}({', '.join(fn.get('parameters', {}).get('properties', {}).keys())})"
          f"  required={required}")
    print(f"    {fn['description']}")

> **Note on MCP.** This case study uses only function tools. Two other agents in the pack
> (`04-hr-policy-mcp`, `07-incident-response-mcp`) attach an MCP server instead, which matters for
> Foundry specifically because **the optimizer can only rewrite `instructions.md` — it cannot touch
> an MCP server's own tool descriptions.** Retrieval behavior for an MCP-backed agent has to be
> governed entirely by the instructions text. Worth a look as a follow-up exercise once you've been
> through this series once with the function-tools-only case study.

## `dataset/` — the Optimize wizard's training material

Two files, and the split between them is itself a Foundry-optimizer-specific feature worth
understanding before you touch the wizard.

In [ ]:
optimize_rows = [json.loads(l) for l in (AGENT_DIR / "dataset" / "optimize.jsonl").read_text().splitlines() if l.strip()]
holdout_rows = [json.loads(l) for l in (AGENT_DIR / "dataset" / "holdout.jsonl").read_text().splitlines() if l.strip()]

print(f"optimize.jsonl: {len(optimize_rows)} rows -- the ONLY file ever uploaded to the wizard")
print(json.dumps(optimize_rows[0], indent=2))
print()
print(f"holdout.jsonl : {len(holdout_rows)} rows -- NEVER uploaded; used only to score the winning candidate afterwards")
print(json.dumps(holdout_rows[0], indent=2))

Why the split exists: if you optimize against a dataset and then "prove" the optimized agent is
better by scoring it on the *same* dataset, the optimizer is being graded on a test it already saw
the answers to. `_tools/build_foundry_dataset.py` (notebook 03) enforces this at the tooling level —
it refuses to touch a file literally named `holdout.jsonl`, not just by convention.

## `expected/expectations.json` — the evaluation contract

This file is **not** a native Foundry artifact — it's this pack's own machine-checkable contract,
used by `_tools/validate_candidate.py` (notebooks 05-06). It models the same content you'd configure
by hand in the wizard's **Criteria** step (the rubric questions an eval model scores a response
against), plus a few things the wizard doesn't have a step for at all (a leakage-safe dataset split,
judge vendor-family pinning, a max cost-growth ratio).

In [ ]:
expectations = json.loads((AGENT_DIR / "expected" / "expectations.json").read_text())
print("Top-level sections:", list(expectations.keys()))
print()
print("judge_config  ->", json.dumps(expectations["judge_config"], indent=2))
print()
print("scoring       ->", json.dumps(expectations["scoring"], indent=2))

In [ ]:
must_have = expectations["instruction_rules"]["must_have"]
print(f"{len(must_have)} must_have rules -- content the optimized candidate must keep, in substance or verbatim:")
for r in must_have:
    print(f"  [{r['id']}] ({r['severity']}) {r['description']}")

## Next

You now know what's going into Foundry. Continue to **`02_create_agent_in_foundry_portal.ipynb`** to
actually create this agent in the Foundry portal.